In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3
import datetime as dt
from tqdm import tqdm
import shutil
from datetime import datetime, timedelta
import re

try:
    import optbinning
except:
    ! pip install optbinning

try:
    import catboost
except:
    ! pip install catboost

In [2]:
dtm_now = dt.datetime.now()
print(f'Latest run date: {dtm_now}')

Latest run date: 2025-03-10 20:55:16.532002


#### Functions

In [3]:
def str_list_to_list(str_list):
    list_out = list(map(int, re.findall(r'\d+', str_list)))
    return list_out

In [4]:
def apply_chime_penalty(flt_ecnl, flt_factor_chime, flt_factor_nonchime, has_inst_tag):
    if has_inst_tag == 1:
        return flt_ecnl * flt_factor_chime
    else:
        return flt_ecnl * flt_factor_nonchime

#### Constants

In [5]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# output
str_dirname_output = './output'

flt_factor_24_to_72 = 2.36

int_n_payments = 4

flt_factor_chime = 1.203
flt_factor_nonchime = 0.936

str_bad_pmt_hx = 'min' # min is optimistic, max is conservative

flt_approval_rate_original = 0.20

Project: 20250307-funded-trends
Task: 09_gen13_simulations


#### Make output dir

In [6]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [7]:
#str_filename = 'df.gzip'
#str_uri = f's3://{str_project}/07_get_predictions/{str_filename}'
#str_uri = f's3://20250121-gen-13-model-monitoring/09_compare_account/{str_filename}'
str_uri = 's3://20250121-gen-13-model-monitoring/07_get_predictions_gen13/df_predictions.gzip'
df = pd.read_parquet(str_uri)
# set dtm
df['request_datetime'] = pd.to_datetime(df['request_datetime'])
# sort
df.sort_values(by='request_datetime', ascending=True, inplace=True)
# make request month
df['request_month'] = df['request_datetime'].apply(
    lambda x: str(x)[:7],
)
# rename
dict_rename = {
    'PD': 'gen13_pd',
    'LGD': 'gen13_lgd',
}
df.rename(columns=dict_rename, inplace=True)
# show
df

,accountid,request_datetime,response_model_name,file_name,bitdebtor,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dealerstate__app,...,g251c__tu_binned_contribution,g095s__tu_binned_contribution,s209a__tu_binned_contribution,inquirynonshortterm12month__ln_binned_contribution,log_odds,gen13_pd,LGD_bk,LGD_nobk,gen13_lgd,request_month
0,8427704,2024-11-27 00:00:11-07:00,PRESTIGE-GEN-XII,8427704/w7Hr-2024-11-27-00:00:11,1,8427704104055851,8427704,10405585,1,Idaho,...,0.002340,0.059176,0.018359,-0.061136,-1.901087,0.129986,0.161799,0.277756,0.277756,2024-11
1,8427704,2024-11-27 00:00:11-07:00,PRESTIGE-GEN-XII,8427704/w7Hr-2024-11-27-00:00:11,0,8427704104055860,8427704,10405586,0,Idaho,...,0.022693,0.059176,-0.120138,-0.061136,-2.636751,0.066810,0.161799,0.277756,0.277756,2024-11
3,8427707,2024-11-27 00:00:41-07:00,PRESTIGE-GEN-XII,8427707/q369-2024-11-27-00:00:41,1,8427707104055891,8427707,10405589,1,Ohio,...,0.022693,0.059176,0.018359,-0.061136,-0.188720,0.452960,0.172898,0.277756,0.277756,2024-11
4,8427708,2024-11-27 00:01:12-07:00,PRESTIGE-GEN-XII,8427708/8SAE-2024-11-27-00:01:12,1,8427708104055901,8427708,10405590,1,Alabama,...,0.022693,-0.142548,0.018359,0.033394,-0.370309,0.408466,0.259641,0.337281,0.337281,2024-11
5,8427715,2024-11-27 00:02:33-07:00,PRESTIGE-GEN-XII,8427715/69b4-2024-11-27-00:02:33,1,8427715104056001,8427715,10405600,1,Oklahoma,...,0.002340,0.059176,0.018359,0.033394,0.879430,0.706704,0.259641,0.337281,0.337281,2024-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9678,8748389,2025-03-09 23:37:28-06:00,PRESTIGE-GEN-XIII,8748389/ws6p-2025-03-09-23:37:28,1,8748389107778621,8748389,10777862,1,California,...,0.022693,0.059176,0.018359,0.033394,1.046202,0.740045,0.259641,0.350185,0.259641,2025-03
9679,8748390,2025-03-09 23:39:37-06:00,PRESTIGE-GEN-XIII,8748390/bl2a-2025-03-09-23:39:37,1,8748390107778631,8748390,10777863,1,Oregon,...,0.022693,-0.142548,0.018359,0.033394,-0.060079,0.484985,0.172898,0.277756,0.277756,2025-03
9680,8748293,2025-03-09 23:40:12-06:00,PRESTIGE-GEN-XIII,8748293/0tM9-2025-03-09-23:40:12,1,8748293107777451,8748293,10777745,1,Washington,...,0.022693,0.059176,0.018359,0.033394,-0.612424,0.351506,0.259641,0.337281,0.337281,2025-03
9681,8748391,2025-03-09 23:46:39-06:00,PRESTIGE-GEN-XIII,8748391/x8IZ-2025-03-09-23:46:39,1,8748391107778641,8748391,10777864,1,Hawaii,...,-0.128272,0.059176,-0.120138,0.033394,-1.024445,0.264162,0.323245,0.390043,0.390043,2025-03


#### Preview

In [8]:
list_cols = [
    'list_pmt_hx_closed__tu_pmthx',
    'list_pmt_hx_open__tu_pmthx',
]
df[list_cols]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx
0,None,[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1...
1,[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1],[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1...
3,[1 1 0 0 1 1 1 1 1 1 1 1 0 0 0 0 0],None
4,[1 1 1 1 1 1 1 1 1 1 1 1 0],None
5,[],None
...,...,...
9678,[1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 0 1 1 1 0 0 0 0...,None
9679,[1 1 1 1 0],None
9680,None,[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1...
9681,[1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 0 0 0 1 0 0 1 1...,None


#### Convert None to string list

In [9]:
for col in tqdm(list_cols):
    df[col] = df[col].apply(
        lambda x: '[]' if x == 'None' else x,
    )
df[list_cols]

100%|██████████| 2/2 [00:00<00:00, 31.41it/s]


,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx
0,[],[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1...
1,[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1],[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1...
3,[1 1 0 0 1 1 1 1 1 1 1 1 0 0 0 0 0],[]
4,[1 1 1 1 1 1 1 1 1 1 1 1 0],[]
5,[],[]
...,...,...
9678,[1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 0 1 1 1 0 0 0 0...,[]
9679,[1 1 1 1 0],[]
9680,[],[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1...
9681,[1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 0 0 0 1 0 0 1 1...,[]


#### Convert to lists

In [10]:
for col in tqdm(list_cols):
    df[col] = df[col].apply(str_list_to_list)
df[list_cols]

100%|██████████| 2/2 [00:02<00:00,  1.19s/it]


,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[]
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[]
5,[],[]
...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[]
9679,"[1, 1, 1, 1, 0]",[]
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[]


#### Create one long list

In [11]:
df['list_pmt_hx'] = df.apply(
    lambda x: x['list_pmt_hx_closed__tu_pmthx'] + x['list_pmt_hx_open__tu_pmthx'],
    axis=1,
)
df[list_cols + ['list_pmt_hx']]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx,list_pmt_hx
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[],"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ..."
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]"
5,[],[],[]
...,...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ..."
9679,"[1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 0]"
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ..."


#### Get length of list

In [12]:
df['n_pmts'] = df['list_pmt_hx'].apply(
    lambda x: len(x),
)
df[list_cols + ['list_pmt_hx', 'n_pmts']]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx,list_pmt_hx,n_pmts
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",46
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",67
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[],"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",17
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",13
5,[],[],[],0
...,...,...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",33
9679,"[1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 0]",5
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",38
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",25


#### Tag

In [13]:
df['tag_min_pmts'] = df['n_pmts'].apply(
    lambda x: 1 if x >= int_n_payments else 0,
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts']]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx,list_pmt_hx,n_pmts,tag_min_pmts
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",46,1
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",67,1
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[],"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",17,1
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",13,1
5,[],[],[],0,0
...,...,...,...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",33,1
9679,"[1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 0]",5,1
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",38,1
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",25,1


#### Get sum of last n payments

In [14]:
df['list_last_n_pmts'] = df['list_pmt_hx'].apply(
    lambda x: x[-int_n_payments:],
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts']]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx,list_pmt_hx,n_pmts,tag_min_pmts,list_last_n_pmts
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",46,1,"[1, 1, 1, 1]"
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",67,1,"[1, 1, 1, 1]"
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[],"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",17,1,"[0, 0, 0, 0]"
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",13,1,"[1, 1, 1, 0]"
5,[],[],[],0,0,[]
...,...,...,...,...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",33,1,"[0, 0, 0, 0]"
9679,"[1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 0]",5,1,"[1, 1, 1, 0]"
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",38,1,"[1, 1, 1, 1]"
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",25,1,"[1, 1, 0, 0]"


#### Get length of list n pmts

In [15]:
df['len_last_n_pmts'] = df['list_last_n_pmts'].apply(
    lambda x: len(x),
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts', 'len_last_n_pmts']]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx,list_pmt_hx,n_pmts,tag_min_pmts,list_last_n_pmts,len_last_n_pmts
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",46,1,"[1, 1, 1, 1]",4
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",67,1,"[1, 1, 1, 1]",4
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[],"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",17,1,"[0, 0, 0, 0]",4
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",13,1,"[1, 1, 1, 0]",4
5,[],[],[],0,0,[],0
...,...,...,...,...,...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",33,1,"[0, 0, 0, 0]",4
9679,"[1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 0]",5,1,"[1, 1, 1, 0]",4
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",38,1,"[1, 1, 1, 1]",4
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",25,1,"[1, 1, 0, 0]",4


#### Get sum

In [16]:
df['sum_last_n_pmts'] = df['list_last_n_pmts'].apply(
    lambda x: np.sum(x),
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts', 'len_last_n_pmts', 'sum_last_n_pmts']]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx,list_pmt_hx,n_pmts,tag_min_pmts,list_last_n_pmts,len_last_n_pmts,sum_last_n_pmts
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",46,1,"[1, 1, 1, 1]",4,4.0
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",67,1,"[1, 1, 1, 1]",4,4.0
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[],"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",17,1,"[0, 0, 0, 0]",4,0.0
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",13,1,"[1, 1, 1, 0]",4,3.0
5,[],[],[],0,0,[],0,0.0
...,...,...,...,...,...,...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",33,1,"[0, 0, 0, 0]",4,0.0
9679,"[1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 0]",5,1,"[1, 1, 1, 0]",4,3.0
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",38,1,"[1, 1, 1, 1]",4,4.0
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",25,1,"[1, 1, 0, 0]",4,2.0


#### Tag if sum = 0 and non-bk

In [17]:
df['tag_bad_pmt_hx'] = df.apply(
    lambda x: 1 if (x['len_last_n_pmts'] >= int_n_payments) and (x['sum_last_n_pmts'] == 0) and (x['ENG-bk'] == 0) else 0,
    axis=1,
)
df[list_cols + ['list_pmt_hx', 'n_pmts', 'tag_min_pmts', 'list_last_n_pmts', 'len_last_n_pmts', 'sum_last_n_pmts', 'ENG-bk', 'tag_bad_pmt_hx']]

,list_pmt_hx_closed__tu_pmthx,list_pmt_hx_open__tu_pmthx,list_pmt_hx,n_pmts,tag_min_pmts,list_last_n_pmts,len_last_n_pmts,sum_last_n_pmts,ENG-bk,tag_bad_pmt_hx
0,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",46,1,"[1, 1, 1, 1]",4,4.0,0,0
1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",67,1,"[1, 1, 1, 1]",4,4.0,0,0
3,"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",[],"[1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, ...",17,1,"[0, 0, 0, 0]",4,0.0,0,1
4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]",13,1,"[1, 1, 1, 0]",4,3.0,0,0
5,[],[],[],0,0,[],0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...
9678,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, ...",33,1,"[0, 0, 0, 0]",4,0.0,1,0
9679,"[1, 1, 1, 1, 0]",[],"[1, 1, 1, 1, 0]",5,1,"[1, 1, 1, 0]",4,3.0,0,0
9680,[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",38,1,"[1, 1, 1, 1]",4,4.0,0,0
9681,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",[],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, ...",25,1,"[1, 1, 0, 0]",4,2.0,0,0


#### Group by account

In [18]:
df_tmp = df.groupby(by='accountid', as_index=False).agg({
    'request_datetime': 'first',
    'ENG-bk': 'first',
    'gen13_pd': 'mean',
    'gen13_lgd': 'mean',
    'has_inst_tag': 'max',
    'tag_bad_pmt_hx': str_bad_pmt_hx,
})
# show
df_tmp

,accountid,request_datetime,ENG-bk,gen13_pd,gen13_lgd,has_inst_tag,tag_bad_pmt_hx
0,5782431,2024-12-19 18:09:20-07:00,0,0.541495,0.390043,0,1
1,7861802,2024-12-04 18:28:18-07:00,0,0.585708,0.337281,1,1
2,7986128,2024-12-27 21:55:29-07:00,1,0.627577,0.259641,0,0
3,8033187,2025-01-03 23:04:31-07:00,1,0.346536,0.293804,0,0
4,8155274,2024-12-23 21:20:25-07:00,1,0.397763,0.316725,0,0
...,...,...,...,...,...,...,...
167415,8748388,2025-03-09 23:36:44-06:00,0,0.249410,0.199397,1,0
167416,8748389,2025-03-09 23:37:28-06:00,1,0.740045,0.259641,1,0
167417,8748390,2025-03-09 23:39:37-06:00,0,0.484985,0.277756,1,0
167418,8748391,2025-03-09 23:46:39-06:00,0,0.264162,0.390043,0,0


#### Get ECNL

In [19]:
df_tmp['gen13_ecnl'] = df_tmp['gen13_pd'] * df_tmp['gen13_lgd'] * flt_factor_24_to_72
# show
df_tmp

,accountid,request_datetime,ENG-bk,gen13_pd,gen13_lgd,has_inst_tag,tag_bad_pmt_hx,gen13_ecnl
0,5782431,2024-12-19 18:09:20-07:00,0,0.541495,0.390043,0,1,0.498446
1,7861802,2024-12-04 18:28:18-07:00,0,0.585708,0.337281,1,1,0.466214
2,7986128,2024-12-27 21:55:29-07:00,1,0.627577,0.259641,0,0,0.384549
3,8033187,2025-01-03 23:04:31-07:00,1,0.346536,0.293804,0,0,0.240280
4,8155274,2024-12-23 21:20:25-07:00,1,0.397763,0.316725,0,0,0.297317
...,...,...,...,...,...,...,...,...
167415,8748388,2025-03-09 23:36:44-06:00,0,0.249410,0.199397,1,0,0.117367
167416,8748389,2025-03-09 23:37:28-06:00,1,0.740045,0.259641,1,0,0.453465
167417,8748390,2025-03-09 23:39:37-06:00,0,0.484985,0.277756,1,0,0.317910
167418,8748391,2025-03-09 23:46:39-06:00,0,0.264162,0.390043,0,0,0.243162


#### Chime factor

In [20]:
df_tmp['gen13_ecnl_chime'] = df_tmp.apply(
    lambda x: apply_chime_penalty(
        flt_ecnl=x['gen13_ecnl'],
        flt_factor_chime=flt_factor_chime,
        flt_factor_nonchime=flt_factor_nonchime,
        has_inst_tag=x['has_inst_tag'],
    ),
    axis=1,
)
# show
df_tmp

,accountid,request_datetime,ENG-bk,gen13_pd,gen13_lgd,has_inst_tag,tag_bad_pmt_hx,gen13_ecnl,gen13_ecnl_chime
0,5782431,2024-12-19 18:09:20-07:00,0,0.541495,0.390043,0,1,0.498446,0.466545
1,7861802,2024-12-04 18:28:18-07:00,0,0.585708,0.337281,1,1,0.466214,0.560855
2,7986128,2024-12-27 21:55:29-07:00,1,0.627577,0.259641,0,0,0.384549,0.359938
3,8033187,2025-01-03 23:04:31-07:00,1,0.346536,0.293804,0,0,0.240280,0.224902
4,8155274,2024-12-23 21:20:25-07:00,1,0.397763,0.316725,0,0,0.297317,0.278288
...,...,...,...,...,...,...,...,...,...
167415,8748388,2025-03-09 23:36:44-06:00,0,0.249410,0.199397,1,0,0.117367,0.141192
167416,8748389,2025-03-09 23:37:28-06:00,1,0.740045,0.259641,1,0,0.453465,0.545518
167417,8748390,2025-03-09 23:39:37-06:00,0,0.484985,0.277756,1,0,0.317910,0.382446
167418,8748391,2025-03-09 23:46:39-06:00,0,0.264162,0.390043,0,0,0.243162,0.227599


#### Auto decision bad pmt hx

In [21]:
df_tmp['gen13_ecnl_chime_pmthx'] = df_tmp.apply(
    lambda x: 1.0 if (x['tag_bad_pmt_hx'] == 1) else x['gen13_ecnl_chime'],
    axis=1,
)
# show
df_tmp

,accountid,request_datetime,ENG-bk,gen13_pd,gen13_lgd,has_inst_tag,tag_bad_pmt_hx,gen13_ecnl,gen13_ecnl_chime,gen13_ecnl_chime_pmthx
0,5782431,2024-12-19 18:09:20-07:00,0,0.541495,0.390043,0,1,0.498446,0.466545,1.000000
1,7861802,2024-12-04 18:28:18-07:00,0,0.585708,0.337281,1,1,0.466214,0.560855,1.000000
2,7986128,2024-12-27 21:55:29-07:00,1,0.627577,0.259641,0,0,0.384549,0.359938,0.359938
3,8033187,2025-01-03 23:04:31-07:00,1,0.346536,0.293804,0,0,0.240280,0.224902,0.224902
4,8155274,2024-12-23 21:20:25-07:00,1,0.397763,0.316725,0,0,0.297317,0.278288,0.278288
...,...,...,...,...,...,...,...,...,...,...
167415,8748388,2025-03-09 23:36:44-06:00,0,0.249410,0.199397,1,0,0.117367,0.141192,0.141192
167416,8748389,2025-03-09 23:37:28-06:00,1,0.740045,0.259641,1,0,0.453465,0.545518,0.545518
167417,8748390,2025-03-09 23:39:37-06:00,0,0.484985,0.277756,1,0,0.317910,0.382446,0.382446
167418,8748391,2025-03-09 23:46:39-06:00,0,0.264162,0.390043,0,0,0.243162,0.227599,0.227599


#### Approved tags

In [22]:
df_tmp['tag_approved_gen13'] = df_tmp['gen13_ecnl'].apply(
    lambda x: 1 if x <= 0.35 else 0,
)
flt_mn_original = df_tmp['tag_approved_gen13'].mean()
print(f'Proportion Approved Gen 13 at {flt_approval_rate_original*100:0.2f}% approval rate: {flt_mn_original:0.4f}')

Proportion Approved Gen 13 at 20.00% approval rate: 0.4526


In [23]:
df_tmp['tag_approved_gen13_chime'] = df_tmp['gen13_ecnl_chime'].apply(
    lambda x: 1 if x <= 0.35 else 0,
)
flt_mn = df_tmp['tag_approved_gen13_chime'].mean()
flt_mn = (flt_mn * flt_approval_rate_original) / flt_mn_original
print(f'Percent Approved Gen 13 with Chime Penalty: {flt_mn*100:0.4f}%')

Percent Approved Gen 13 with Chime Penalty: 18.7179%


In [24]:
df_tmp['tag_approved_gen13_chime_pmthx'] = df_tmp['gen13_ecnl_chime_pmthx'].apply(
    lambda x: 1 if x <= 0.35 else 0,
)
flt_mn = df_tmp['tag_approved_gen13_chime_pmthx'].mean()
flt_mn = (flt_mn * flt_approval_rate_original) / flt_mn_original
print(f'Percent Approved Gen 13 with Chime Penalty and Pmt Hx Penalty: {flt_mn*100:0.4f}%')

Percent Approved Gen 13 with Chime Penalty and Pmt Hx Penalty: 16.0274%
